### Chunking the loaded files

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_code(content, chunk_size=500, overlap=50):
    """
    Split code content into overlapping chunks.
    
    Args:
        content (str): Python code as a string
        chunk_size (int): Maximum number of characters per chunk
        overlap (int): Number of characters to overlap between chunks
    
    Returns:
        List[str]: List of code chunks
    """
    chunks = []
    start = 0
    while start < len(content):
        end = start + chunk_size
        chunks.append(content[start:end])
        start += chunk_size - overlap
    return chunks

def chunk_content(file_content, file_type):
    """
    Chunk file content based on type.
    
    - Markdown / documentation files are split according to structure to preserve semantic meaning.
    - Code files are split by size with overlap, since small blocks retain meaningful context for embeddings.
    
    Args:
        file_content (str): The content of the file
        file_type (str): 'markdown' or other (code)
    
    Returns:
        List[str]: List of text or code chunks
    """

    if file_type == "markdown":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            separators=["\n\n", "\n", " "]
        )
        return splitter.split_text(file_content)
    else:
        return chunk_code(file_content, chunk_size=500, overlap=50)


        



In [ ]:
chunk_content(""""# Power Consumption Prediction App
<div>
<img width="1699" height="930" alt="Screenshot 2026-01-04 144915" src="https://github.com/user-attachments/assets/d3e701eb-a958-4c20-9a40-bea9894f02f9" />
<img width="1687" height="893" alt="Screenshot 2026-01-04 144932" src="https://github.com/user-attachments/assets/f91e1054-0f70-4858-9e21-dc81338dcf41" />
</div>
This project predicts household power consumption using historical electrical measurements and time-based features.  
It includes a trained machine learning model, a Streamlit web application, and a Docker setup for easy deployment.

---

## Overview

The goal of this project is to predict **Global Active Power (kW)** using:
- Electrical sensor readings
- Past power usage (lag features)
- Time information such as hour, day of week, and month

The model is trained on the *Individual Household Electric Power Consumption* dataset.

---

## Features Used

- Global reactive power  
- Voltage  
- Global intensity  
- Sub-metering (1, 2, 3)  
- Lag features (`lag_1`, `lag_2`)  
- Hour of day  
- Day of week  
- Month  

Lag features help the model capture time-based patterns in electricity usage.

---

## Model Details

- **Algorithm:** Random Forest Regressor  
- **Preprocessing:**
  - Median imputation for missing values
  - Feature selection using Mutual Information
- **Train/Test Split:** Time-based split to avoid data leakage

---

## Model Performance

Performance on unseen test data:

- **MAE:** ~0.0177  
- **RMSE:** ~0.0329  
- **R² Score:** ~0.9987  

---

## Streamlit Application

The app provides:
- Single power consumption prediction
- Batch prediction using CSV upload
- Automatic lag feature creation
- Error metric display (MAE) when actual values are available

---

## Running Locally
```bash
Install dependencies:
pip install -r requirements.txt


Run the app:
streamlit run app/app.py

Running with Docker

Build the image:
docker build -t power-consumption-app .


Run the container:
docker run -p 5000:8501 power-consumption-app


Open:
http://localhost:5000



""","markdown")

['"# Power Consumption Prediction App\n<div>\n<img width="1699" height="930" alt="Screenshot 2026-01-04 144915" src="https://github.com/user-attachments/assets/d3e701eb-a958-4c20-9a40-bea9894f02f9" />\n<img width="1687" height="893" alt="Screenshot 2026-01-04 144932" src="https://github.com/user-attachments/assets/f91e1054-0f70-4858-9e21-dc81338dcf41" />\n</div>\nThis project predicts household power consumption using historical electrical measurements and time-based features.  \nIt includes a trained machine learning model, a Streamlit web application, and a Docker setup for easy deployment.\n\n---\n\n## Overview\n\nThe goal of this project is to predict **Global Active Power (kW)** using:\n- Electrical sensor readings\n- Past power usage (lag features)\n- Time information such as hour, day of week, and month\n\nThe model is trained on the *Individual Household Electric Power Consumption* dataset.\n\n---\n\n## Features Used',
 '---\n\n## Features Used\n\n- Global reactive power  \n- V

In [42]:
chunk_code("""import streamlit as st
import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error
import os

BASE_DIR = os.path.dirname(__file__) 
MODEL_PATH = os.path.join(BASE_DIR, "power_consumption_model.pkl")

@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

model = load_model()

st.title(" Power Consumption Predictor")

num_cols = [
    'Global_reactive_power','Voltage','Global_intensity',
    'Sub_metering_1','Sub_metering_2','Sub_metering_3',
    'lag_1','lag_2','hour','day_of_week','month'
]

col1, col2 = st.columns(2)

 
with col1:
    st.header("Single Prediction")

    inputs = {
        'Global_reactive_power': st.number_input("Global Reactive Power", 0.0, 1.0),
        'Voltage': st.number_input("Voltage", 200.0, 260.0),
        'Global_intensity': st.number_input("Global Intensity", 0.0, 50.0),
        'Sub_metering_1': st.number_input("Sub_metering_1", 0.0, 50.0),
        'Sub_metering_2': st.number_input("Sub_metering_2", 0.0, 50.0),
        'Sub_metering_3': st.number_input("Sub_metering_3", 0.0, 50.0),
        'lag_1': st.number_input("Lag 1", 0.0, 10.0),
        'lag_2': st.number_input("Lag 2", 0.0, 10.0),
        'hour': st.slider("Hour", 0, 23),
        'day_of_week': st.slider("Day of Week (0=Mon)", 0, 6),
        'month': st.slider("Month", 1, 12)
    }

    if st.button("Predict"):
        sample = pd.DataFrame([inputs])
        pred = model.predict(sample)[0]
        st.success(f"Predicted Power: {pred:.3f} kW")
 
with col2:
    st.header("Batch Prediction")

    file = st.file_uploader("Upload CSV", type="csv")
    if st.button("Predict Batch", key="batch_predict"):

        if file:
            df = pd.read_csv(file)

            # Ensure Datetime exists
            if {'Date', 'Time'}.issubset(df.columns):
                df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
                df = df.sort_values('Datetime')

                # Time features
                df['hour'] = df['Datetime'].dt.hour
                df['day_of_week'] = df['Datetime'].dt.dayofweek
                df['month'] = df['Datetime'].dt.month
            else:
                st.error("CSV must contain Date and Time columns")
                st.stop()
                
            if df.empty:
                st.error("No valid rows left after creating lag features.")
                st.stop()

            # Create lag features (SAME AS TRAINING)
            if 'Global_active_power' in df.columns:
                df['lag_1'] = df['Global_active_power'].shift(1)
                df['lag_2'] = df['Global_active_power'].shift(2)
            else:
                st.error("Global_active_power required to create lag features")
                st.stop()

            # Drop rows with missing lags
            df = df.dropna(subset=['lag_1', 'lag_2'])
     
            # Predict
            preds = model.predict(df)
            df["Predicted_Power"] = preds

            # Metrics if actual exists
            if 'Global_active_power' in df.columns:
                mae = mean_absolute_error(df['Global_active_power'], preds)
                st.metric("MAE", f"{mae:.4f}")

            st.dataframe(df)
     
""",200,50)

['import streamlit as st\nimport pandas as pd\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import mean_absolute_error\nimport os\n\nBASE_DIR = os.path.dirname(__file__) \nMODEL_PATH = os.path.join(B',
 'ath.dirname(__file__) \nMODEL_PATH = os.path.join(BASE_DIR, "power_consumption_model.pkl")\n\n@st.cache_resource\ndef load_model():\n    return joblib.load(MODEL_PATH)\n\nmodel = load_model()\n\nst.title(" Pow',
 '(MODEL_PATH)\n\nmodel = load_model()\n\nst.title(" Power Consumption Predictor")\n\nnum_cols = [\n    \'Global_reactive_power\',\'Voltage\',\'Global_intensity\',\n    \'Sub_metering_1\',\'Sub_metering_2\',\'Sub_metering',
 '   \'Sub_metering_1\',\'Sub_metering_2\',\'Sub_metering_3\',\n    \'lag_1\',\'lag_2\',\'hour\',\'day_of_week\',\'month\'\n]\n\ncol1, col2 = st.columns(2)\n\n\nwith col1:\n    st.header("Single Prediction")\n\n    inputs = {\n  ',
 ' st.header("Single Prediction")\n\n    inputs = {\n        \'Global_reactive_power\': st.number_input("Global React